# 01 — Data loading

This notebook loads the raw IST file, finds and fixes its encoding problem, and writes a UTF-8 copy that the following notebooks use.

**Input:** `data/raw/IST_corrected.csv` (version 2 of the IST database, never modified).  
**Output:** `data/interim/IST_corrected_utf8.csv`.

1. First attempt: reading the raw file fails
2. Inspecting the raw bytes
3. Conversion to UTF-8
4. Loading the converted file
5. Column types
6. Overview of all columns

## Setup

In [ ]:
import re
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

RAW_PATH = Path("../data/raw/IST_corrected.csv")
UTF8_PATH = Path("../data/interim/IST_corrected_utf8.csv")

con = duckdb.connect()

## 1. First attempt: reading the raw file fails

Reading the raw file directly raises an error. The cell catches it and prints its first line, so that the notebook still runs from top to bottom.

In [ ]:
try:
    con.sql(f"""
        CREATE OR REPLACE VIEW ist_raw AS
        SELECT * FROM read_csv_auto('{RAW_PATH.as_posix()}')
    """)
except duckdb.InvalidInputException as error:
    print("DuckDB error:", str(error).splitlines()[0])

DuckDB expects UTF-8 and finds an invalid byte sequence: the file has a different encoding. Ignoring the errors (`ignore_errors=true`) would silently drop rows, so we inspect the bytes instead.

## 2. Inspecting the raw bytes

In [ ]:
raw = RAW_PATH.read_bytes()

print("size:", len(raw), "bytes")
print("start of file:", raw[:200])
print(
    "line endings \\r\\n:",
    raw.count(b"\r\n"),
    " \\n:",
    raw.count(b"\n"),
    " \\r:",
    raw.count(b"\r"),
)

non_ascii = sorted({b for b in raw if b > 127})
print("non-ASCII bytes:", [hex(b) for b in non_ascii])

for match in list(re.finditer(rb"[\x80-\xff]+", raw))[:5]:
    print(raw[max(0, match.start() - 15) : match.end() + 15])

**Findings**

- The first line is a header with the column names (`HOSPNUM`, `RDELAY`, …).
- 19,436 lines, all ending with `\r\n` (Windows): 1 header + 19,435 patients, as stated in the paper.
- The only non-ASCII byte is `0x9F`, always inside `pa\x9f-91`: the Polish month abbreviation `paź` (October). In **Windows-1250** (Central European encoding), `0x9F` is `ź`.

Decoding with the wrong encoding (e.g. Latin-1) would raise no error but would silently produce a wrong character: hence the `assert` below.

## 3. Conversion to UTF-8

The raw file is left untouched; the converted copy goes to `data/interim`. `write_bytes` is used instead of `write_text` so that only the encoding changes, not the line endings.

In [ ]:
text = raw.decode("cp1250")
assert "paź" in text, "unexpected decoding: check the encoding"

UTF8_PATH.write_bytes(text.encode("utf-8"))
print("written:", UTF8_PATH, UTF8_PATH.stat().st_size, "bytes")

## 4. Loading the converted file

In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ist AS
    SELECT * FROM read_csv_auto('{UTF8_PATH.as_posix()}')
""")
con.sql("SELECT COUNT(*) AS n_patients FROM ist").df()

19,435 patients, as expected. Column names and types inferred by DuckDB:

In [ ]:
con.sql("DESCRIBE ist").df()

## 5. Column types

Number of columns of each type, in SQL and in pandas (the two results must agree).

In [ ]:
con.sql("""
    SELECT column_type, COUNT(*) AS n_columns
    FROM (DESCRIBE ist)
    GROUP BY column_type
    ORDER BY n_columns DESC
""").df()

In [ ]:
schema = con.sql("DESCRIBE ist").df()
schema["column_type"].value_counts()

## 6. Overview of all columns

`SUMMARIZE` gives, for each column, its type, min, max, approximate number of distinct values, mean, quartiles and percentage of missing values. The detailed checks are in `02_data_checking.ipynb`.

Note: for the free-text comment columns, min and max are fragments of individual records, so clear this output before committing.

In [ ]:
con.sql("SUMMARIZE ist").df()

## Summary

- The raw file is encoded in **Windows-1250**: the only non-ASCII character is `ź` in the Polish month abbreviation `paź` (column `RDATE`).
- It is converted to UTF-8 in `data/interim/IST_corrected_utf8.csv`; the raw file is not modified.
- The converted file loads correctly: 19,435 patients, 112 columns.